In [1]:
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from utils import *
from sklearn.metrics import roc_auc_score

In [2]:
train_feats = pd.read_csv('../Data/train_features.csv')
features_dict = pickle.load(open('../Files/all_features.pkl', 'rb'))

In [3]:
all_features = list(set(sum(features_dict.values(), [])) - set(features_dict['identifier_features']))

### Descriptive statistics

In [4]:
null_df = pd.DataFrame({
    'null': train_feats[all_features].isna().sum(),
    'null_rate': train_feats[all_features].isna().sum()/train_feats.shape[0],
    'std': train_feats[all_features].std()
})
null_df.sort_values(by='null_rate', ascending=False)

,null,null_rate,std
debt_to_ebitda,73229,0.154040,18265.573864
operating_margin,15583,0.032779,22546.372596
interest_coverage_proxy,15469,0.032540,32009.208549
current_assets_to_debt_st,952,0.002003,2358.670469
cash_to_debt_st,952,0.002003,940.865557
bank_debt_concentration,189,0.000398,222.958660
st_debt_share,189,0.000398,0.320855
cash_flow_to_debt,189,0.000398,63.995016
operating_cashflow_to_liabilities,93,0.000196,24.463025
legal_struct_SAU,0,0.000000,0.000000


### Univariate Analysis

In [5]:
def univariate_analysis(X, y, feature_categories):
    X = X.replace([np.inf, -np.inf], np.nan)
    y = pd.Series(np.asarray(y), index=X.index)

    results = []

    for category, features in feature_categories.items():
        print(category)
        for feature in tqdm(features):
            x = X[feature]
            valid = x.notna()

            if x[valid].nunique() < 2 or y[valid].nunique() < 2: continue

            auc = roc_auc_score(y[valid], x[valid])

            results.append({
                "category": category,
                "feature": feature,
                "auc": auc,
                "auc_strength": max(auc, 1 - auc),
                "missing_pct": 100 * (1 - valid.mean()),
                "n_observed": valid.sum()
            })
    return (
        pd.DataFrame(results)
        .sort_values(["category", "auc_strength"], ascending=[True, False])
        .reset_index(drop=True)
    )

In [6]:
feature_categories = {
    'size': ["log_assets"],
    'profitability': ["net_profit_to_assets","operating_margin","ebitda_to_assets","roa"],
    'liquidity': ["working_capital_to_assets","cash_to_assets","cash_to_debt_st","current_assets_to_asset_total","current_assets_to_debt_st"],
    'solvency': ["equity_to_assets","debt_to_assets","debt_st_to_assets","st_debt_share","bank_debt_concentration"],
    'debt_servicing': ["interest_coverage_proxy","cash_flow_to_debt","debt_to_ebitda"],
    'cash_flow': ["operating_cashflow_to_assets","operating_cashflow_to_liabilities","cashflow_profit_gap_to_assets"],
    'efficiency': ["sales_to_assets","receivables_to_assets","account_payables_to_assets"],
    'flags': features_dict['flag_features'],
    'categorical_feats': features_dict['categorical_features']
}
univariate_analysis_results = univariate_analysis(train_feats, train_feats['default'], feature_categories)

size


100%|██████████| 1/1 [00:00<00:00,  6.86it/s]


profitability


100%|██████████| 4/4 [00:00<00:00,  6.10it/s]


liquidity


100%|██████████| 5/5 [00:00<00:00, 10.67it/s]


... (8 outputs left out)

flags


100%|██████████| 10/10 [00:00<00:00, 17.28it/s]


categorical_feats


100%|██████████| 6/6 [00:00<00:00, 187.91it/s]


In [7]:
univariate_analysis_results

,category,feature,auc,auc_strength,missing_pct,n_observed
0,cash_flow,operating_cashflow_to_liabilities,0.258507,0.741493,0.019563,475296
1,cash_flow,operating_cashflow_to_assets,0.266384,0.733616,0.000000,475389
2,cash_flow,cashflow_profit_gap_to_assets,0.519475,0.519475,0.000000,475389
3,debt_servicing,cash_flow_to_debt,0.258406,0.741594,0.039757,475200
4,debt_servicing,interest_coverage_proxy,0.294960,0.705040,3.253967,459920
5,debt_servicing,debt_to_ebitda,0.647694,0.647694,15.404017,402160
6,efficiency,receivables_to_assets,0.601546,0.601546,0.000000,475389
7,efficiency,account_payables_to_assets,0.590998,0.590998,0.000000,475389
8,efficiency,sales_to_assets,0.504973,0.504973,0.000000,475389
9,flags,net_loss_flag,0.654812,0.654812,0.000000,475389


In [8]:
selected_feats = list(univariate_analysis_results.query('auc_strength >= 0.6')['feature'])

In [9]:
train_feats[feature_categories['categorical_feats']].nunique()

legal_struct_SAA    1
legal_struct_SAU    1
legal_struct_SPA    1
legal_struct_SRL    1
legal_struct_SRS    1
legal_struct_SRU    1
dtype: int64

### Correlation

In [10]:
def drop_corr_features(df, corr_feat_vars, threshold, univariate_analysis_results):
    corr_df = df[corr_feat_vars].corr()
    corr_pairs = corr_df.unstack()

    high_corr = corr_pairs[(abs(corr_pairs) > threshold) & (corr_pairs != 1.0)]
    high_corr = high_corr.drop_duplicates()
    print(high_corr)
    high_cor_vars = list(dict(high_corr).keys())
    print(high_cor_vars)

    # select the feature giving highest lift (precision per bin)
    def features_to_drop(corr_pairs):
        return univariate_analysis_results.query(f'feature in @corr_pairs').sort_values(by='auc_strength', ascending=True).iloc[0]['feature']
        
    corr_feats_to_drop = [features_to_drop(corr_pairs) for corr_pairs in high_cor_vars]
    return corr_feats_to_drop

In [11]:
high_corr_vars = drop_corr_features(train_feats, selected_feats, 0.9, univariate_analysis_results)
high_corr_vars

ebitda_to_assets  roa               0.999831
equity_to_assets  debt_to_assets   -0.972868
dtype: float64
[('ebitda_to_assets', 'roa'), ('equity_to_assets', 'debt_to_assets')]


['roa', 'debt_to_assets']

In [12]:
selected_feats = [c for c in selected_feats if c not in high_corr_vars]
selected_feats

['operating_cashflow_to_liabilities',
 'operating_cashflow_to_assets',
 'cash_flow_to_debt',
 'interest_coverage_proxy',
 'debt_to_ebitda',
 'receivables_to_assets',
 'net_loss_flag',
 'operating_loss_flag',
 'nonpositive_ebitda_flag',
 'negative_equity_flag',
 'cash_to_debt_st',
 'current_assets_to_debt_st',
 'working_capital_to_assets',
 'net_profit_to_assets',
 'ebitda_to_assets',
 'operating_margin',
 'equity_to_assets',
 'debt_st_to_assets']

### VIF (Impute, create flag and compute)

In [13]:
na_df = pd.DataFrame(train_feats[selected_feats].isna().sum())
na_df.columns = ['na_cnt']
cols_to_impute = list(na_df.query('na_cnt > 0').index)
cols_to_impute

['operating_cashflow_to_liabilities',
 'cash_flow_to_debt',
 'interest_coverage_proxy',
 'debt_to_ebitda',
 'cash_to_debt_st',
 'current_assets_to_debt_st',
 'operating_margin']

In [15]:
feature_imputer = median_table_by_ateco_sector(train_feats, cols_to_impute, asset_col='log_assets')
feature_imputer.to_csv('../Data/feature_imputer.csv', index=False)
feature_imputer = pd.read_csv('../Data/feature_imputer.csv')
feature_imputer

,ateco_sector,operating_cashflow_to_liabilities,cash_flow_to_debt,interest_coverage_proxy,debt_to_ebitda,cash_to_debt_st,current_assets_to_debt_st,operating_margin,log_assets
0,1.0,0.000632,0.000655,0.021097,1.068389,0.001508,0.066114,0.001182,15.107622
1,2.0,0.000515,0.000515,0.070402,1.068880,0.001960,0.071297,0.002147,14.801390
2,3.0,0.002411,0.002672,0.072338,0.693964,0.000908,0.079880,0.002306,14.869053
3,5.0,0.002931,0.002988,0.131004,0.529735,0.005588,0.076568,0.002343,16.948253
4,6.0,-0.000393,-0.000611,-0.524460,0.111748,0.002112,0.070636,-0.015309,16.029307
...,...,...,...,...,...,...,...,...,...
79,94.0,0.004271,0.004356,0.109898,0.616032,0.000836,0.066341,0.009517,15.278666
80,95.0,0.002757,0.003195,0.180154,0.614747,0.005018,0.080883,0.002649,14.747473
81,96.0,0.003926,0.004581,0.095277,0.464172,0.002625,0.059582,0.002977,15.056277
82,99.0,0.000040,0.000040,0.052839,1.753276,0.001077,0.089747,0.004751,14.795285


In [16]:
def missing_value_imputation(df, imputation_table, cols_to_impute, asset_col='asst_tot'):
    df_copy = df.copy()
    for c in df.columns:
        if c in cols_to_impute: df[c] = pd.to_numeric(df[c], errors="coerce")

    # Create missingness flags before imputation
    flag_cols = {col: f"{col}_imputed_flag" for col in [asset_col, *cols_to_impute]}
    for col, flag_col in flag_cols.items(): df_copy[flag_col] = df_copy[col].isna().astype(int)

    imputation_table = imputation_table.set_index("ateco_sector")

    row_medians = imputation_table.reindex(df_copy["ateco_sector"]).set_axis(df_copy.index)
    row_medians = row_medians.fillna(imputation_table.loc["global"])
    assets = df_copy[asset_col].fillna(row_medians[asset_col])

    df_copy[cols_to_impute] = df_copy[cols_to_impute].fillna(row_medians[cols_to_impute].mul(assets, axis=0))

    return df_copy

In [17]:
train_feats_imputed = missing_value_imputation(train_feats, feature_imputer, cols_to_impute, asset_col='log_assets')

In [18]:
train_feats_imputed[selected_feats].isna().sum()

operating_cashflow_to_liabilities    0
operating_cashflow_to_assets         0
cash_flow_to_debt                    0
interest_coverage_proxy              0
debt_to_ebitda                       0
receivables_to_assets                0
net_loss_flag                        0
operating_loss_flag                  0
nonpositive_ebitda_flag              0
negative_equity_flag                 0
cash_to_debt_st                      0
current_assets_to_debt_st            0
working_capital_to_assets            0
net_profit_to_assets                 0
ebitda_to_assets                     0
operating_margin                     0
equity_to_assets                     0
debt_st_to_assets                    0
dtype: int64

In [37]:
flag_corr = train_feats_imputed[[c for c in train_feats_imputed.columns if 'flag' in c]].corr()

# Find nearly perfectly correlated flags
for i, col1 in enumerate(imputed_flag_cols):
    for col2 in imputed_flag_cols[i+1:]:
        corr = flag_corr.loc[col1, col2]

        if pd.notna(corr) and abs(corr) > 0.999:
            print(col1, "<-->", col2, "Correlation:", corr)

cash_to_debt_st_imputed_flag <--> current_assets_to_debt_st_imputed_flag Correlation: 1.0


In [38]:
# imputed_flag_cols
reduced_flag_cols = ['cash_to_debt_st_imputed_flag', 'operating_cashflow_to_liabilities_imputed_flag', 'interest_coverage_proxy_imputed_flag', 'operating_margin_imputed_flag']

In [39]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

imputed_flag_cols = [f"{col}_imputed_flag" for col in cols_to_impute]

X_vif = sm.add_constant(train_feats_imputed[selected_feats + reduced_flag_cols])
vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.to_numpy(), i) for i in range(X_vif.shape[1])
    ],
})

vif.sort_values("VIF", ascending=False)

,feature,VIF
8,operating_loss_flag,3.297599
9,nonpositive_ebitda_flag,3.039712
18,debt_st_to_assets,2.877178
17,equity_to_assets,1.812351
13,working_capital_to_assets,1.713624
7,net_loss_flag,1.552789
6,receivables_to_assets,1.491508
1,operating_cashflow_to_liabilities,1.385908
11,cash_to_debt_st,1.332261
12,current_assets_to_debt_st,1.272679


In [40]:
vif.to_csv('../Files/VIF.csv', index=False)

### Save the final list of features

In [31]:
pickle.dump({'final_selected_features': selected_feats}, open('../Files/selected_features.csv', 'wb'))

In [32]:
pickle.load(open('../Files/selected_features.csv', 'rb'))

{'final_selected_features': ['operating_cashflow_to_liabilities',
  'operating_cashflow_to_assets',
  'cash_flow_to_debt',
  'interest_coverage_proxy',
  'debt_to_ebitda',
  'receivables_to_assets',
  'net_loss_flag',
  'operating_loss_flag',
  'nonpositive_ebitda_flag',
  'negative_equity_flag',
  'cash_to_debt_st',
  'current_assets_to_debt_st',
  'working_capital_to_assets',
  'net_profit_to_assets',
  'ebitda_to_assets',
  'operating_margin',
  'equity_to_assets',
  'debt_st_to_assets']}